# Despliegue — Trufi App Cochabamba
## CRISP-DM Fase 6 · Productos por lotes

Entrada: `reports/04_evaluacion/predicciones_cruzadas.parquet` (predicciones fuera de pliegue de la técnica
adoptada, R5). Salidas en `outputs/`: predicciones por celda (CSV + GeoJSON), mapa HTML, celdas prioritarias,
ficha del modelo y README de actualización. Decisiones: `reports/05_despliegue/DECISIONES_05_despliegue.md`.

In [1]:
import sys
from pathlib import Path


def _encontrar_raiz(inicio: Path) -> Path:
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "pyproject.toml").exists():
            return carpeta
    raise RuntimeError("No se encontró pyproject.toml")


RAIZ = _encontrar_raiz(Path.cwd())
SRC_DIR = RAIZ / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [2]:
from trufi_ds.notebook_setup import *  # noqa: F403
from trufi_ds import preparation as prep
from trufi_ds.config import PLAZA_14_SEPTIEMBRE

import json
import branca.colormap as cm
import folium
import geopandas as gpd

OUT = PROJECT_ROOT / "outputs"
OUT.mkdir(exist_ok=True)
EVAL = PROJECT_ROOT / "reports" / "04_evaluacion"
MODELADO = PROJECT_ROOT / "reports" / "03_modelado"
PREP = PROJECT_ROOT / "reports" / "02_preparacion"

pred = pd.read_parquet(EVAL / "predicciones_cruzadas.parquet")
prueba = pd.read_csv(EVAL / "resultados_prueba.csv")
adopcion = json.loads((MODELADO / "adopcion.json").read_text())
MODEL_DATE = prueba["fecha"].iloc[0][:10]
print(f"{len(pred):,} celdas · técnica {adopcion['adopted']} · fecha del modelo {MODEL_DATE}")

1,381 celdas · técnica B1 · fecha del modelo 2026-09-27


## D1 · Predicciones por celda (CSV y GeoJSON EPSG:4326)

In [3]:
COLUMNAS = ["h3_cell", "lat", "lon", "population", "query_count", "expected_count", "p_low", "pearson_residual",
            "gap_category", "gtfs_covered", "dist_stop_m", "in_test", "model_id", "model_date", "edge_cell"]
salida = pred.assign(model_date=MODEL_DATE)[COLUMNAS].sort_values("h3_cell").reset_index(drop=True)
salida.round({"lat": 6, "lon": 6, "expected_count": 3, "p_low": 5, "pearson_residual": 4, "dist_stop_m": 1}).to_csv(
    OUT / "predictions_by_cell.csv", index=False)

gdf = gpd.GeoDataFrame(salida, geometry=prep.cell_polygons(salida["h3_cell"].tolist()), crs="EPSG:4326").to_crs("EPSG:4326")
gdf["queries_per_1000"] = gdf["query_count"] / gdf["population"] * 1000
gdf.to_file(OUT / "predictions_by_cell.geojson", driver="GeoJSON", COORDINATE_PRECISION=6)
print(salida["gap_category"].value_counts().to_dict())
salida.head()

{'esperado': 1253, 'exceso': 105, 'bajo_lo_esperado': 23}


,h3_cell,lat,lon,population,query_count,expected_count,p_low,pearson_residual,gap_category,gtfs_covered,dist_stop_m,in_test,model_id,model_date,edge_cell
0,888b2c8001fffff,-17.429585,-66.270150,253,35,178.990099,0.410139,-0.498660,esperado,0,1733.937172,False,B1,2026-09-27,False
1,888b2c8003fffff,-17.422968,-66.275400,10,3,10.466483,0.500358,-0.434748,esperado,0,873.263865,False,B1,2026-09-27,False
2,888b2c8005fffff,-17.428167,-66.261581,200,0,137.702146,0.103826,-0.619671,bajo_lo_esperado,0,1792.601596,False,B1,2026-09-27,False
3,888b2c8007fffff,-17.421551,-66.266830,69,4,67.704937,0.276772,-0.582222,esperado,0,1049.374055,False,B1,2026-09-27,False
4,888b2c8009fffff,-17.437619,-66.273470,137,1,2.186170,0.638960,-0.310457,esperado,0,1985.070562,False,B1,2026-09-27,False


## D3 · Celdas prioritarias (D-402)

20 celdas con `population` ≥ mediana: primero las de `deficit` (5 %), luego las de `bajo_lo_esperado` (D-310) y,
si faltan, las de menor `p_low`. Dentro de cada grupo, por `p_low` creciente. La columna `note` explica cada caso.

In [4]:
mediana_pob = salida["population"].median()
candidatas = salida[salida["population"] >= mediana_pob].copy()
candidatas["significant_deficit"] = candidatas["gap_category"] == "deficit"
orden_cat = {"deficit": 0, "bajo_lo_esperado": 1}
candidatas["_orden"] = candidatas["gap_category"].map(orden_cat).fillna(2)
NOTAS = {0: "déficit significativo (p_low < 0,05)",
         1: "bajo lo esperado (p_low < 0,20, categoría descriptiva D-310)",
         2: "candidata a revisar: menor p_low, sin categoría de déficit (D-402)"}
prioritarias = (candidatas.sort_values(["_orden", "p_low"]).head(20)
                .assign(rank=lambda d: range(1, len(d) + 1),
                        osm_url=lambda d: [f"https://www.openstreetmap.org/?mlat={a:.5f}&mlon={b:.5f}#map=16/{a:.5f}/{b:.5f}"
                                           for a, b in zip(d["lat"], d["lon"])],
                        note=lambda d: d["_orden"].map(NOTAS)))
prioritarias = prioritarias[["rank", "h3_cell", "lat", "lon", "population", "query_count", "expected_count", "p_low",
                             "pearson_residual", "gap_category", "significant_deficit", "gtfs_covered", "dist_stop_m",
                             "edge_cell", "note", "osm_url"]]
prioritarias.round({"expected_count": 2, "p_low": 4, "pearson_residual": 4, "dist_stop_m": 0}).to_csv(OUT / "priority_cells.csv", index=False)
print(f"Mediana de población: {mediana_pob:.0f} · categorías de las prioritarias: {prioritarias['gap_category'].value_counts().to_dict()}")
prioritarias.head(10)

Mediana de población: 388 · categorías de las prioritarias: {'bajo_lo_esperado': 14, 'esperado': 6}


,rank,h3_cell,lat,lon,population,query_count,expected_count,p_low,pearson_residual,gap_category,significant_deficit,gtfs_covered,dist_stop_m,edge_cell,note,osm_url
97,1,888b2c8119fffff,-17.415877,-66.232560,392,8,858.362626,0.132877,-0.614615,bajo_lo_esperado,False,0,897.890640,False,"bajo lo esperado (p_low < 0,20, categoría desc...",https://www.openstreetmap.org/?mlat=-17.41588&...
86,2,888b2c8101fffff,-17.419652,-66.210182,1587,28,2737.304252,0.134633,-0.614146,bajo_lo_esperado,False,1,162.097986,False,"bajo lo esperado (p_low < 0,20, categoría desc...",https://www.openstreetmap.org/?mlat=-17.41965&...
341,3,888b2c880bfffff,-17.354905,-66.135631,734,27,2495.567181,0.137611,-0.613776,bajo_lo_esperado,False,0,973.096756,False,"bajo lo esperado (p_low < 0,20, categoría desc...",https://www.openstreetmap.org/?mlat=-17.35490&...
25,4,888b2c803dfffff,-17.413516,-66.263511,794,23,2010.618252,0.140821,-0.613379,bajo_lo_esperado,False,0,523.523536,False,"bajo lo esperado (p_low < 0,20, categoría desc...",https://www.openstreetmap.org/?mlat=-17.41352&...
89,5,888b2c8107fffff,-17.411619,-66.206866,1911,56,4115.211201,0.149521,-0.612064,bajo_lo_esperado,False,0,613.171478,False,"bajo lo esperado (p_low < 0,20, categoría desc...",https://www.openstreetmap.org/?mlat=-17.41162&...
1254,6,888b2cd6a7fffff,-17.427074,-66.029961,487,0,46.890633,0.156877,-0.618004,bajo_lo_esperado,False,0,567.990085,False,"bajo lo esperado (p_low < 0,20, categoría desc...",https://www.openstreetmap.org/?mlat=-17.42707&...
364,7,888b2c8865fffff,-17.363865,-66.099484,653,8,523.152738,0.160669,-0.610823,bajo_lo_esperado,False,1,438.642829,False,"bajo lo esperado (p_low < 0,20, categoría desc...",https://www.openstreetmap.org/?mlat=-17.36386&...
104,8,888b2c8129fffff,-17.416812,-66.193056,1788,99,5439.357019,0.166814,-0.609221,bajo_lo_esperado,False,1,405.642193,False,"bajo lo esperado (p_low < 0,20, categoría desc...",https://www.openstreetmap.org/?mlat=-17.41681&...
13,9,888b2c8021fffff,-17.418714,-66.249694,491,10,516.875231,0.174729,-0.608305,bajo_lo_esperado,False,0,1022.615754,False,"bajo lo esperado (p_low < 0,20, categoría desc...",https://www.openstreetmap.org/?mlat=-17.41871&...
99,10,888b2c811dfffff,-17.414458,-66.223995,716,43,1940.694878,0.180480,-0.606727,bajo_lo_esperado,False,1,45.814424,False,"bajo lo esperado (p_low < 0,20, categoría desc...",https://www.openstreetmap.org/?mlat=-17.41446&...


## D2 · Mapa HTML (D-403)

In [5]:
area = gpd.read_file(PREP / "area_estudio.geojson")
polos = pd.read_csv(PREP / "polos_actividad.csv")
centro = list(prep.area_center(area.geometry.iloc[0]))
mapa = folium.Map(location=centro, zoom_start=11, tiles="OpenStreetMap", control_scale=True)

base = gdf[["h3_cell", "population", "query_count", "expected_count", "pearson_residual", "p_low", "gap_category",
            "gtfs_covered", "dist_stop_m", "queries_per_1000", "geometry"]].copy()
base = base.round({"expected_count": 1, "pearson_residual": 3, "p_low": 3, "dist_stop_m": 0, "queries_per_1000": 1})
tooltip = folium.GeoJsonTooltip(fields=["h3_cell", "population", "query_count", "expected_count", "pearson_residual",
                                        "p_low", "gap_category", "gtfs_covered", "dist_stop_m"],
                                aliases=["Celda", "Población", "Consultas", "Esperadas", "Residuo", "p_low", "Categoría",
                                         "Cubierta ≤500 m", "Parada más cercana (m)"])


def capa_continua(nombre: str, columna: str, colormap, mostrar: bool = False) -> None:
    datos = base.copy()
    datos["_color"] = [colormap(v) for v in datos[columna]]
    folium.GeoJson(
        datos.to_crs("EPSG:4326").to_json(), name=nombre, show=mostrar,
        style_function=lambda f: {"fillColor": f["properties"]["_color"], "color": "none", "fillOpacity": 0.75},
        tooltip=folium.GeoJsonTooltip(fields=tooltip.fields, aliases=tooltip.aliases),
    ).add_to(mapa)


log_tasa = np.log10(base["queries_per_1000"] + 1)
cm_tasa = cm.LinearColormap(["#f7fbff", "#6baed6", "#08306b"], vmin=0, vmax=float(log_tasa.quantile(0.98)),
                            caption="log10(consultas por 1.000 hab. + 1)")
base["_log_tasa"] = log_tasa
capa_continua("Consultas por 1.000 hab.", "_log_tasa", cm_tasa, mostrar=False)

log_esp = np.log10(base["expected_count"] + 1)
cm_esp = cm.LinearColormap(["#fff5eb", "#fd8d3c", "#7f2704"], vmin=0, vmax=float(log_esp.quantile(0.98)),
                           caption="log10(consultas esperadas + 1)")
base["_log_esp"] = log_esp
capa_continua("Consultas esperadas", "_log_esp", cm_esp)

lim = float(np.percentile(np.abs(base["pearson_residual"]), 98))
lim_neg = min(-0.5, float(np.percentile(base["pearson_residual"], 2)))
cm_res = cm.LinearColormap(["#b2182b", "#f7f7f7", "#2166ac"], index=[lim_neg, 0, lim], vmin=lim_neg, vmax=lim,
                           caption="Residuo de Pearson NB (rojo: menos consultas de lo esperado; azul: más)")
capa_continua("Residuo (brecha)", "pearson_residual", cm_res, mostrar=True)
cm_res.add_to(mapa)

cubiertas = base[base["gtfs_covered"] == 1]
folium.GeoJson(cubiertas.to_crs("EPSG:4326").to_json(), name="Celdas cubiertas (parada ≤ 500 m)", show=False,
               style_function=lambda f: {"fillColor": "#1b9e77", "color": "#1b9e77", "weight": 0.3, "fillOpacity": 0.35}).add_to(mapa)

grupo_prio = folium.FeatureGroup(name="Celdas prioritarias (D-402)", show=True)
for r in prioritarias.itertuples():
    folium.CircleMarker([r.lat, r.lon], radius=7, color="#d73027", fill=True, fill_opacity=0.9,
                        popup=folium.Popup(f"<b>#{r.rank}</b> {r.h3_cell}<br>Población {r.population:,}<br>Consultas {r.query_count} "
                                           f"(esperadas {r.expected_count:.1f})<br>p_low {r.p_low:.3f}<br>{r.note}<br>"
                                           f"<a href='{r.osm_url}' target='_blank'>OpenStreetMap</a>", max_width=300)).add_to(grupo_prio)
grupo_prio.add_to(mapa)

grupo_polos = folium.FeatureGroup(name="Polos de actividad (población 0)", show=False)
for r in polos.itertuples():
    folium.CircleMarker([r.lat, r.lon], radius=4, color="#6a3d9a", fill=True, fill_opacity=0.8,
                        tooltip=f"{r.h3_cell}: {r.query_count} consultas, sin población Kontur").add_to(grupo_polos)
grupo_polos.add_to(mapa)

folium.GeoJson(area.to_crs("EPSG:4326").to_json(), name="Área de estudio",
               style_function=lambda f: {"fill": False, "color": "#424242", "weight": 1.5}).add_to(mapa)
folium.Marker(centro, tooltip="Centro del área (D-022)", icon=folium.Icon(color="black", icon="star")).add_to(mapa)
folium.Marker([PLAZA_14_SEPTIEMBRE["lat"], PLAZA_14_SEPTIEMBRE["lon"]], tooltip="Plaza 14 de Septiembre", icon=folium.Icon(color="gray", icon="info-sign")).add_to(mapa)
folium.LayerControl(collapsed=False).add_to(mapa)
aviso = ("<div style='position:fixed;bottom:18px;left:18px;z-index:9999;background:white;padding:8px 10px;"
         "border:1px solid #999;font:12px sans-serif;max-width:360px'>"
         f"<b>Brecha de consultas Trufi App</b> · técnica {adopcion['adopted']} · {MODEL_DATE}<br>"
         f"Predicciones fuera de pliegue. Celdas: déficit (5 %) {(salida['gap_category'] == 'deficit').mean()*100:.1f} %, "
         f"bajo lo esperado (D-310) {(salida['gap_category'] == 'bajo_lo_esperado').mean()*100:.1f} %, "
         f"exceso {(salida['gap_category'] == 'exceso').mean()*100:.1f} %. "
         "Las celdas prioritarias son candidatas a revisar. Asociaciones, no causas.</div>")
mapa.get_root().html.add_child(folium.Element(aviso))
mapa.save(OUT / "gap_map.html")
print(f"gap_map.html: {(OUT / 'gap_map.html').stat().st_size / 1e6:.1f} MB")

gap_map.html: 3.6 MB


## D4 · Ficha del modelo

In [6]:
resumen_cv = pd.read_csv(MODELADO / "resumen_cv.csv").set_index("model")
rp = prueba[prueba["distance_ring"] == "todas"].set_index("model")
sens = pd.read_csv(EVAL / "sensibilidad.csv")
cob = pd.read_csv(EVAL / "brecha_por_cobertura.csv").set_index("umbral_m").loc[500]
moran = pd.read_csv(EVAL / "moran_residuos.csv").iloc[0]
estructura = pd.read_csv(EVAL / "residuos_estructura.csv")
flujo = pd.read_csv(PREP / "tabla_flujo_limpieza.csv")
cmp_area = dict(zip(*pd.read_csv(PREP / "area_comparacion_etapas.csv").T.values))
particion = pd.read_csv(PREP / "particion_resumen.csv").set_index("conjunto")
A = adopcion["adopted"]
reparto = pd.read_csv(EVAL / "reparto_brecha.csv").set_index("gap_category")["pct"]
pct = lambda c: float(reparto.get(c, 0.0))

NOMBRES = {"B0": "tasa global × población", "B0.5": "tasa del municipio × población",
           "B0.7": "tasa del anillo de distancia × población", "B1": "tasa de las celdas vecinas × población",
           "M1": "GLM Poisson con offset de población", "M2": "Binomial Negativa con offset de población",
           "M3": "gradient boosting con pérdida Poisson"}
LIMITES = {
    "B1": ["**Copia la vecindad.** Predice con la tasa de las celdas de entrenamiento cercanas: una zona entera con poca "
           "adopción de la app parecerá \"esperada\", porque sus vecinas también consultan poco. La brecha que detecta es local "
           "(una celda frente a su entorno), no regional.",
           "**No explica por qué.** No usa variables: no dice si la demanda se debe a la población, la distancia o la actividad.",
           "**Depende del tamaño de vecindad** (`min_neighbors = 3`, `k_max = 10`); ver `b1_sensibilidad_k.csv`.",
           "**Zonas aisladas y borde.** Si no hay celdas de entrenamiento cerca, usa la tasa global (B0), y las celdas de borde "
           "tienen menos vecinas."],
    "B0.5": ["**Una tasa por municipio**: ignora las diferencias dentro del municipio (centro frente a periferia)."],
    "B0.7": ["**Una tasa por anillo**: supone que la demanda depende solo de la distancia al centro."],
    "B0": ["**Una sola tasa**: supone que todas las celdas consultan igual por habitante."],
    "M1": ["**Relación log-lineal fija** y errores estándar poco fiables con la sobredispersión observada."],
    "M2": ["**Relación log-lineal fija**; extrapola la tendencia de distancia a zonas sin datos."],
    "M3": ["**Caja negra**: la relación se describe con dependencia parcial; puede memorizar celdas (se usan predicciones fuera de pliegue)."],
}
rho_txt = "; ".join(f"{r.grupo}: ρ = {r.valor:+.2f}" for r in estructura.query("analisis == 'spearman_residuo_vs_variable'").itertuples())

def cv(m: str, met: str, nd: int = 3) -> str:
    return f"{resumen_cv.loc[m, f'{met}_mean']:.{nd}f} ± {resumen_cv.loc[m, f'{met}_std']:.{nd}f}"

filas_tabla = "\n".join(
    f"| {'**' + m + '**' if m == A else m} | {NOMBRES[m]}{' (adoptada)' if m == A else ''} | {cv(m, 'poisson_deviance', 0)} | {cv(m, 'd2')} | "
    + (f"{rp.loc[m, 'poisson_deviance']:.1f}" if m in rp.index else "—") + " |" for m in resumen_cv.index)
lim_tecnica = "\n".join(f"- {t}" for t in LIMITES.get(A, []))
bloques_prueba, pct_cons_prueba = int(particion.loc["prueba", "bloques"]), float(particion.loc["prueba", "pct_consultas"])

model_card = f"""# Ficha del modelo — Demanda esperada de consultas Trufi App por celda H3

**Versión:** {MODEL_DATE} (iteración 2) · **Técnica adoptada:** {A} — {NOMBRES[A]} · **Repositorio:** `trufi-data-science`

## Qué hace
Estima el número esperado de consultas de ruta de Trufi App que se originan en cada celda H3 (resolución 8, ~0,74 km²)
del área de estudio del eje metropolitano de Cochabamba, y lo compara con las consultas observadas. La diferencia
(residuo) es la *brecha*.

## Datos
- **Consultas**: {int(flujo['filas_despues'].iloc[-1]):,} consultas válidas (de {int(flujo['filas_antes'].iloc[0]):,} crudas), 12-sep-2022 a 9-jun-2024, con 7 semanas sin datos en 2024 (no imputadas). Se descartan los viajes de más de 50 km (D-020).
- **Población**: Kontur Population 2023 (H3 r8). Sensibilidad con la edición 2022.
- **Área**: envolvente de los orígenes válidos (componente espacial principal) + 1 km, {float(cmp_area['area_etapa2_hull_origenes_km2']):,.0f} km², sin depender del filtro de distancia (D-021). Incluye el valle alto.
- **Centro de referencia**: centroide del área (D-022), a {float(cmp_area['distancia_centro_a_plaza_km']):.1f} km de la Plaza 14 de Septiembre.
- **Unidad**: {len(salida):,} celdas con población ≥ 10.
- **GTFS**: solo para el contraste posterior; **no** entra al modelo.

## Técnica y cómo se eligió
Se compararon siete técnicas con validación cruzada espacial (`GroupKFold(5)` por bloque H3 res 6) y una regla
declarada antes de ver resultados (> 5 % de mejora en devianza y 4 de 5 pliegues ganados). Cuatro son **líneas base**
(reglas simples sin variables: B0, B0.5, B0.7, B1), dos son **modelos estadísticos** (M1, M2) y una de **machine
learning** (M3). La explicación completa está en `reports/03_modelado/decision_adopcion.md`.

| Técnica | Qué hace | Devianza (CV espacial) | D² (CV espacial) | Devianza prueba |
|---|---|---|---|---|
{filas_tabla}

Prueba: {bloques_prueba} bloques reservados antes de modelar ({pct_cons_prueba:.1f} % de las consultas), evaluados una sola vez;
D² de {A} = {rp.loc[A, 'd2']:.3f}, calibración (Σŷ/Σy) = {rp.loc[A, 'calibration']:.2f}, Spearman = {rp.loc[A, 'spearman']:.3f}.

## Productos
- `predictions_by_cell.csv` / `.geojson`: una fila por celda con `expected_count`, `pearson_residual`, `p_low`, `gap_category`.
- `gap_map.html`: mapa interactivo con capas de tasa, esperadas, residuo, cobertura, prioritarias y polos.
- `priority_cells.csv`: 20 celdas candidatas a revisar.

## Limitaciones de la técnica adoptada ({A})
{lim_tecnica}
- **Estructura que queda en los residuos** (`residuos_estructura.csv`): {rho_txt}.

## Limitaciones generales (léanse antes de usar)
1. **Déficit estadístico.** Celdas en `deficit` (p < 0,05): {pct('deficit'):.1f} %. En `bajo_lo_esperado` (p_low < 0,20, categoría descriptiva D-310): {pct('bajo_lo_esperado'):.1f} %. En `exceso`: {pct('exceso'):.1f} %. Con la sobredispersión observada, el déficit al 5 % es difícil de alcanzar; las celdas prioritarias son candidatas a revisar, no un hallazgo estadístico.
2. **Asociación, no causa.** Que las celdas sin parada GTFS a ≤ 500 m tengan residuos distintos (Cliff δ = {cob['cliff_delta']:+.2f}) no prueba que la falta de rutas reduzca las consultas: Trufi también mapea donde ya hay demanda.
3. **Consultas ≠ población.** Reflejan a quienes usan la app. No se sabe si el origen es la ubicación GPS o un punto elegido en el mapa.
4. **Kontur es una estimación modelada**, no un censo.
5. **Prueba pequeña**: {bloques_prueba} bloques; interprétese con prudencia.
6. **Varianza alta entre pliegues**: el pliegue con el centro domina la devianza media.
7. **Celdas de borde** (`edge_cell = True`): tienen menos vecinas.

## Cuándo **no** usarlo
- Para estimar demanda fuera del área de estudio o en celdas con población < 10.
- Como prueba de que una zona "necesita rutas": úsese para priorizar la **revisión** en terreno.
- Para comparar periodos (el modelo es transversal: suma todo el periodo).
- Con otra edición de Kontur u otro periodo de consultas sin reentrenar (ver `README.md`).

## Estabilidad
El ranking de brecha tiene Spearman ≥ {sens['spearman_brecha_vs_referencia'].dropna().min():.3f} frente a Kontur 2022, filtros de 20/30/40 km,
población ≥ 50, objetivo `user_count` y distancia a la Plaza en lugar del centroide. I de Moran de los residuos = {moran['moran_I']:.3f} (p = {moran['p_sim']:.2f}).

## Trazabilidad
Decisiones: `DECISIONES.md` y `reports/0*/DECISIONES_*.md`. Criterio de éxito: `reports/04_evaluacion/criterio_exito.md`.
Informe de cierre: `docs/INFORME_FINAL.md`. Iteración 1: `reports/_iteracion1/`.
"""
(OUT / "MODEL_CARD.md").write_text(model_card, encoding="utf-8")

5457

## D5 · README de actualización

In [7]:
readme = f"""# outputs/ — Productos de despliegue (Fase 6)

Generados por `notebooks/05_despliegue.ipynb` (versión {MODEL_DATE}). Léase primero `MODEL_CARD.md`.

| Archivo | Contenido |
|---|---|
| `predictions_by_cell.csv` | Una fila por celda H3 r8 del modelo: observado, esperado, residuo, `p_low`, categoría, cobertura GTFS |
| `predictions_by_cell.geojson` | Lo mismo con polígonos H3, EPSG:4326 |
| `gap_map.html` | Mapa interactivo (abrir en un navegador; no requiere servidor) |
| `priority_cells.csv` | 20 celdas candidatas a revisar, con enlace a OpenStreetMap |
| `MODEL_CARD.md` | Ficha del modelo: datos, técnica, métricas, limitaciones |

## Cómo regenerar

Desde la raíz del repositorio, con los datos en `data/` (ver `AGENTS.md`):

```bash
uv sync
for nb in 01_comprension_datos 02_preparacion_datos 03_modelado 04_evaluacion 05_despliegue; do
  uv run jupyter nbconvert --to notebook --execute --inplace notebooks/$nb.ipynb
done
```

`04_evaluacion` no reevalúa la prueba si `reports/04_evaluacion/resultados_prueba.csv` ya existe (D-302). Para una
actualización con datos nuevos, archive ese archivo y declare el cambio en `DECISIONES_04_evaluacion.md` **antes**
de ejecutar.

## Cuándo actualizar

```mermaid
flowchart TD
    K[Nueva edición de Kontur] --> P2[02 Preparación: T4–T8<br/>población, coronas, celdas]
    Q[Un año de consultas nuevas] --> P1[01 Comprensión de datos]
    P1 --> P2
    P2 --> T6{{¿Cambian los bloques?<br/>nueva reserva de prueba + commit}}
    T6 --> P4[03 Modelado: M0 protocolo + CV]
    P4 --> P5[04 Evaluación: prueba única, brecha, E9]
    P5 --> P6[05 Despliegue: outputs/]
    G[Nuevo GTFS] --> C[Solo contraste: dist_stop_m, gtfs_covered<br/>E5 + capa de cobertura]
    C --> P6
```

| Disparador | Qué se rehace | Por qué |
|---|---|---|
| Nueva edición de Kontur | Etapa 2 (T4–T8) y Fases 4–6 | Cambian la exposición y las coronas de población |
| Nuevo GTFS | Solo `dist_stop_m`, `gtfs_covered`, E5 y la capa de cobertura | GTFS no entra al modelo (D-017) |
| Un año de consultas nuevas | Todo, desde la Etapa 1 | Cambian el objetivo, el área (D-018) y posiblemente los bloques |

## Publicación opcional (GitHub Pages)

No se publicó (D-405). Para hacerlo: copiar `gap_map.html` a `docs/index.html` (o a una rama `gh-pages`), activar
Pages en *Settings → Pages* del repositorio y añadir aquí el enlace. Revíselo antes: el mapa muestra conteos por celda
de una app de terceros.
"""
(OUT / "README.md").write_text(readme, encoding="utf-8")
print(sorted(p.name for p in OUT.iterdir()))

['MODEL_CARD.md', 'README.md', 'gap_map.html', 'predictions_by_cell.csv', 'predictions_by_cell.geojson', 'priority_cells.csv']
